In [1]:
%pip install pypdf

  Using cached pypdf-6.21.4-py3-none-any.whl.metadata (7.6 kB)
Using cached pypdf-6.21.4-py3-none-any.whl (410 kB)

[notice] A new release of pip is available: 26.0.1 -> 26.2.2
[notice] To update, run: /opt/anaconda3/bin/python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
from pypdf import PdfReader

reader = PdfReader("invoices/invoice_01.pdf")
text = reader.pages[0].extract_text()
print(text)

Delta Paper & Print Co.
Industrial Zone 3, 10th of Ramadan City
Tax Reg. No. 696-159-619
INVOICE
Invoice No: INV-2026-0407
Date: 02/09/2026
Due Date: 02/10/2026
Bill To:
Nile Office Interiors
14 Abbas El Akkad St, Nasr City, Cairo
Description
Qty
Unit price
Amount
Toner cartridge 85A
1
1,850.00
1,850.00
Lever arch files
50
85.00
4,250.00
A4 copy paper, box of 5 reams
2
950.00
1,900.00
Subtotal
EGP 8,000.00
VAT 14%
EGP 1,120.00
TOTAL DUE
EGP 9,120.00
Bank: National Bank of Egypt, IBAN EG00 0000 0000 0000 0000 0000 000. Thank you for your business.



In [3]:
reader = PdfReader("invoices/invoice_18.pdf")
text = reader.pages[0].extract_text()
print(text)

In [4]:
import os

for filename in sorted(os.listdir("invoices")):
    if not filename.endswith(".pdf"):
        continue
    reader = PdfReader(f"invoices/{filename}")
    text = reader.pages[0].extract_text()
    if text.strip() == "":
        print(filename, "-> NO TEXT (scanned)")
    else:
        print(filename, "->", len(text), "characters")

invoice_01.pdf -> 553 characters
invoice_02.pdf -> 430 characters
invoice_03.pdf -> 584 characters
invoice_04.pdf -> 373 characters
invoice_05.pdf -> 557 characters
invoice_06.pdf -> 432 characters
invoice_07.pdf -> 454 characters
invoice_08.pdf -> 506 characters
invoice_09.pdf -> 519 characters
invoice_10.pdf -> 458 characters
invoice_11.pdf -> 560 characters
invoice_12.pdf -> 396 characters
invoice_13.pdf -> 555 characters
invoice_14.pdf -> 506 characters
invoice_15.pdf -> 511 characters
invoice_16.pdf -> 511 characters
invoice_17.pdf -> 423 characters
invoice_18.pdf -> NO TEXT (scanned)
invoice_19.pdf -> 516 characters
invoice_20.pdf -> 425 characters


In [5]:
%pip install google-genai python-dotenv

  Using cached google_auth-2.62.0-py3-none-any.whl.metadata (6.0 kB)
  Using cached httpx-0.28.1-py3-none-any.whl.metadata (7.1 kB)
  Using cached typing_extensions-4.16.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached typing_inspection-0.4.4-py3-none-any.whl.metadata (2.6 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 3.1 MB/s  0:00:00 eta 0:00:01
Using cached google_auth-2.62.0-py3-none-any.whl (267 kB)
Using cached httpx-0.28.1-py3-none-any.whl (73 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 2.9 MB/s  0:00:00 eta 0:00:01
Using cached typing_extensions-4.16.0-py3-none-any.whl (45 kB)
Using cached typing_inspection-0.4.4-py3-none-any.whl (14 kB)
  Attempting uninstall: typing-extensions
    Found existing installation: typing_extensions 4.15.0
    Uninstalling typing_extensions-4.15.0:
      Successfully uninstalled typing_extensions-4.15.0━━━━━━━━━━━ 1/9 [typing-extensions]
  Attempting uninstall: typing-inspection━━━━━━━━━━━━━━━━━━━━━━━━━ 1/9 [typing-ex

In [7]:
import os
from dotenv import load_dotenv
from google import genai

load_dotenv(".env", override=True)
print("Key found:", os.getenv("GEMINI_API_KEY") is not None)
client = genai.Client()

Key found: True


In [8]:
reader = PdfReader("invoices/invoice_01.pdf")
text = reader.pages[0].extract_text()

prompt = f"""Extract these fields from the invoice below:
invoice number, supplier name, invoice date, due date, total amount, currency.

Invoice text:
{text}"""

response = client.models.generate_content(model="gemini-3.1-flash-lite", contents=prompt)
print(response.text)

Here are the extracted fields from the invoice:

*   **Invoice Number:** INV-2026-0407
*   **Supplier Name:** Delta Paper & Print Co.
*   **Invoice Date:** 02/09/2026
*   **Due Date:** 02/10/2026
*   **Total Amount:** 9,120.00
*   **Currency:** EGP


In [9]:
reader = PdfReader("invoices/invoice_03.pdf")
text = reader.pages[0].extract_text()

prompt = f"""Extract these fields from the invoice below:
invoice number, supplier name, invoice date, due date, total amount, currency.

Invoice text:
{text}"""

response = client.models.generate_content(model="gemini-3.1-flash-lite", contents=prompt)
print(response.text)

Here are the extracted fields from the invoice:

*   **Invoice Number:** SI/26/00093
*   **Supplier Name:** Pyramids Furniture Factory
*   **Invoice Date:** 08-Sep-2026
*   **Due Date:** 08-Oct-2026
*   **Total Amount:** 45,030.00
*   **Currency:** EGP


In [10]:
from pydantic import BaseModel, Field
from typing import Optional

class Invoice(BaseModel):
    invoice_number: Optional[str] = None
    supplier: Optional[str] = None
    invoice_date: Optional[str] = Field(None, description="Format YYYY-MM-DD")
    due_date: Optional[str] = Field(None, description="Format YYYY-MM-DD")
    total: Optional[float] = Field(None, description="Final total as a plain number")
    currency: Optional[str] = Field(None, description="3-letter code like EGP or USD")

In [11]:
def extract(path):
    text = PdfReader(path).pages[0].extract_text()
    response = client.models.generate_content(
        model="gemini-3.1-flash-lite",
        contents=f"Extract the invoice fields from this text:\n\n{text}",
        config={"response_mime_type": "application/json", "response_schema": Invoice},
    )
    return response.parsed

print(extract("invoices/invoice_01.pdf"))
print(extract("invoices/invoice_03.pdf"))

invoice_number='INV-2026-0407' supplier='Delta Paper & Print Co.' invoice_date='2026-09-02' due_date='2026-10-02' total=9120.0 currency='EGP'
invoice_number='SI/26/00093' supplier='Pyramids Furniture Factory' invoice_date='2026-09-08' due_date='2026-10-08' total=45030.0 currency='EGP'


In [12]:
print(extract("invoices/invoice_07.pdf"))
print(extract("invoices/invoice_11.pdf"))

invoice_number='TI-49354' supplier='Zamalek Cleaning Services' invoice_date='2026-09-21' due_date='2026-10-21' total=29001.6 currency='EGP'
invoice_number=None supplier='Cairo Cable Works' invoice_date='2026-10-03' due_date='2026-11-02' total=21072.9 currency='EGP'


In [13]:
from google.genai import types

RULES = """Extract the invoice fields.
Rules:
- Only use values that are printed on the invoice.
- If a field is not printed, return null. Never calculate or guess it.
- Payment terms like "Net 30" are NOT a due date."""

def extract(path):
    text = PdfReader(path).pages[0].extract_text()
    if text.strip() == "":
        # Scanned invoice: no text, so send the PDF itself for Gemini to look at
        pdf_bytes = open(path, "rb").read()
        contents = [types.Part.from_bytes(data=pdf_bytes, mime_type="application/pdf"), RULES]
    else:
        contents = f"{RULES}\n\nInvoice text:\n{text}"
    response = client.models.generate_content(
        model="gemini-3.1-flash-lite",
        contents=contents,
        config={"response_mime_type": "application/json", "response_schema": Invoice},
    )
    return response.parsed

In [14]:
print(extract("invoices/invoice_07.pdf"))
print(extract("invoices/invoice_11.pdf"))
print(extract("invoices/invoice_18.pdf"))

invoice_number='TI-49354' supplier='Zamalek Cleaning Services' invoice_date='2026-09-21' due_date=None total=29001.6 currency='EGP'
invoice_number=None supplier='Cairo Cable Works' invoice_date='2026-10-03' due_date='2026-11-02' total=21072.9 currency='EGP'
invoice_number='INV-2026-0526' supplier='Pyramids Furniture Factory' invoice_date='2026-10-22' due_date='2026-11-21' total=22458.0 currency='EGP'


In [15]:
import time
import pandas as pd

rows = []
for filename in sorted(os.listdir("invoices")):
    if not filename.endswith(".pdf"):
        continue
    print("Reading", filename)
    try:
        inv = extract(f"invoices/{filename}")
    except Exception as e:
        print("   failed, waiting 60 seconds:", str(e)[:80])
        time.sleep(60)
        inv = extract(f"invoices/{filename}")
    row = {"file": filename}
    row.update(inv.model_dump())
    rows.append(row)
    time.sleep(5)

df = pd.DataFrame(rows)
df.to_excel("extracted_invoices.xlsx", index=False)
df

Reading invoice_01.pdf
Reading invoice_02.pdf
Reading invoice_03.pdf
Reading invoice_04.pdf
Reading invoice_05.pdf
   failed, waiting 60 seconds: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently exp
Reading invoice_06.pdf
Reading invoice_07.pdf
Reading invoice_08.pdf
Reading invoice_09.pdf
Reading invoice_10.pdf
Reading invoice_11.pdf
Reading invoice_12.pdf
Reading invoice_13.pdf
Reading invoice_14.pdf
Reading invoice_15.pdf
Reading invoice_16.pdf
Reading invoice_17.pdf
Reading invoice_18.pdf
   failed, waiting 60 seconds: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently exp
Reading invoice_19.pdf
Reading invoice_20.pdf


,file,invoice_number,supplier,invoice_date,due_date,total,currency
0,invoice_01.pdf,INV-2026-0407,Delta Paper & Print Co.,2026-09-02,2026-10-02,9120.00,EGP
1,invoice_02.pdf,26026,CAIRO CABLE WORKS,2026-09-05,2026-09-20,23529.60,EGP
2,invoice_03.pdf,SI/26/00093,Pyramids Furniture Factory,2026-09-08,2026-10-08,45030.00,EGP
3,invoice_04.pdf,TI-64937,Alex Packaging Solutions,2026-09-10,2026-10-25,221.16,EGP
4,invoice_05.pdf,INV-2026-0435,Sinai Steel Fixtures,2026-09-14,2026-10-14,14720.00,EGP
5,invoice_06.pdf,26078,HORUS IT SERVICES,2026-09-17,2026-09-24,69996.00,EGP
6,invoice_07.pdf,TI-49354,Zamalek Cleaning Services,2026-09-21,None,29001.60,EGP
7,invoice_08.pdf,SI/26/00248,Red Sea Logistics,2026-09-24,2026-10-24,55700.00,EGP
8,invoice_09.pdf,INV-2026-0463,Horus IT Services,2026-09-28,2026-10-28,1523.72,USD
9,invoice_10.pdf,26130,DELTA PAPER & PRINT CO.,2026-10-01,2026-10-31,11343.00,EGP


In [18]:
key = pd.read_csv("answer_key/answer_key.csv", dtype=str, keep_default_na=False)
merged = df.merge(key, on="file", suffixes=("_ai", "_key"))

fields = ["invoice_number", "supplier", "invoice_date", "due_date", "total", "currency"]
results = []
for _, r in merged.iterrows():
    for f in fields:
        ai = r[f + "_ai"]
        truth = r[f + "_key"]
        if ai is None or pd.isna(ai):
            ai = ""
        if f == "total":
            correct = abs(float(ai) - float(truth)) < 0.01
        else:
            correct = str(ai) == truth
        results.append({"file": r["file"], "field": f, "ai": ai, "key": truth, "correct": correct})

res = pd.DataFrame(results)
print("Fields correct:", res["correct"].sum(), "of", len(res))
ok = res.groupby("file")["correct"].all()
print("Invoices fully correct:", ok.sum(), "of", len(ok))
res[~res["correct"]]

Fields correct: 120 of 120
Invoices fully correct: 20 of 20


,file,field,ai,key,correct


In [17]:
SUPPLIERS = [
    "Delta Paper & Print Co.", "Cairo Cable Works", "Pyramids Furniture Factory",
    "Alex Packaging Solutions", "Sinai Steel Fixtures", "Horus IT Services",
    "Zamalek Cleaning Services", "Red Sea Logistics",
]

def match_supplier(name):
    if name is None:
        return None
    for s in SUPPLIERS:
        if s.lower() == name.strip().lower():
            return s
    return name

df["supplier"] = df["supplier"].apply(match_supplier)
df.to_excel("extracted_invoices.xlsx", index=False)
df[["file", "supplier"]]

,file,supplier
0,invoice_01.pdf,Delta Paper & Print Co.
1,invoice_02.pdf,Cairo Cable Works
2,invoice_03.pdf,Pyramids Furniture Factory
3,invoice_04.pdf,Alex Packaging Solutions
4,invoice_05.pdf,Sinai Steel Fixtures
5,invoice_06.pdf,Horus IT Services
6,invoice_07.pdf,Zamalek Cleaning Services
7,invoice_08.pdf,Red Sea Logistics
8,invoice_09.pdf,Horus IT Services
9,invoice_10.pdf,Delta Paper & Print Co.
